# 🔬 Notebook 3: S3 — Deep Dives

Four short, runnable deep dives:

1. **Presigned URLs** — grant one-shot access without leaking keys.
2. **Consistent hashing** — pick a storage node without reshuffling
   the world when you add capacity.
3. **Erasure coding** — intuition with XOR parity.
4. **Lifecycle policies** — move cold data to cheap storage (or
   delete it) automatically.

Each dive follows the same shape: show the obvious **bad** approach,
then the **better** one with code.


## 🛠️ Setup

```bash
cd 06-system-designs/s3
uv sync
```

Then in VS Code pick the `.venv` kernel from the top-right of the notebook. If
it doesn't show up: `Cmd+Shift+P` → **Reload Window** and try again.

Everything in this lab is **pure Python** — no databases, no Docker. You can
run it on a laptop in a few seconds.


## 1️⃣ Presigned URLs

### Bad: share the master credentials 😱

Imagine a mobile app that needs to download `photos/cat.jpg`. The
quickest thing to ship: put the AWS access key in the app. This is
a catastrophe:

- No expiry — leak = forever access.
- No scope — leak = full account access.
- Impossible to rotate without shipping a new app version.


### Better: presigned URL with HMAC + expiry 🔐

Idea: the server knows a secret, the client doesn't. The server
computes `sig = HMAC(secret, method|bucket|key|expiry)` and hands the
client a URL that includes the signature and the expiry. The server
re-computes the signature on arrival and compares.

Any tampering (different method, different object, expired clock,
changed signature) makes the re-computed HMAC mismatch, so access is
refused.


In [1]:
import hmac, hashlib, time, urllib.parse

SECRET_BYTES = b"super-secret"   # lives ONLY on the server

def presign(method: str, bucket: str, key: str, ttl: int = 60) -> str:
    expires = int(time.time()) + ttl
    msg = f"{method}|{bucket}|{key}|{expires}".encode()
    sig = hmac.new(SECRET_BYTES, msg, hashlib.sha256).hexdigest()
    return f"/{bucket}/{key}?exp={expires}&sig={sig}"

def verify(method: str, bucket: str, key: str, url: str):
    try:
        qs = urllib.parse.parse_qs(url.split("?", 1)[1])
        exp = int(qs["exp"][0])
        sig = qs["sig"][0]
    except (IndexError, KeyError, ValueError):
        return False, "malformed"
    if time.time() > exp:
        return False, "expired"
    msg = f"{method}|{bucket}|{key}|{exp}".encode()
    expected = hmac.new(SECRET_BYTES, msg, hashlib.sha256).hexdigest()
    # compare_digest is constant-time -> no timing side-channels
    ok = hmac.compare_digest(sig, expected)
    return ok, "ok" if ok else "bad sig"

url = presign("GET", "photos", "cat.jpg", ttl=60)
print("URL:", url)
print("valid GET:      ", verify("GET", "photos", "cat.jpg", url))
print("wrong method:   ", verify("PUT", "photos", "cat.jpg", url))
print("wrong object:   ", verify("GET", "photos", "dog.jpg", url))
print("tampered sig:   ", verify("GET", "photos", "cat.jpg", url[:-1] + "0"))

short = presign("GET", "photos", "cat.jpg", ttl=0)
time.sleep(1)
print("expired:        ", verify("GET", "photos", "cat.jpg", short))


URL: /photos/cat.jpg?exp=1776651990&sig=f3d6d256620dba69514f40301e55619decf64a3fb0a792a3c0a2e95c9c39be11
valid GET:       (True, 'ok')
wrong method:    (False, 'bad sig')
wrong object:    (False, 'bad sig')
tampered sig:    (False, 'bad sig')


expired:         (False, 'expired')


Why this design is nice:

- **No server round-trip for the download itself** — the client hits
  the storage URL directly, saving bandwidth on the API front-end.
- **Scoped** — a leak only buys you *that one object*.
- **Time-bounded** — even a leak is fine after the TTL.
- **Stateless** — the server doesn't remember the URL; it just
  re-derives the signature and compares.


## 2️⃣ Consistent hashing for shard placement

### Bad: `hash(key) % N` 💥

Pick a node for each key with `hash(key) % N`. Simple. Now add one
more node. Suddenly **most** keys map to a different node, so we
reshuffle the entire dataset just to grow by 25%.


In [2]:
def mod_placement(key, nodes):
    return nodes[hash(key) % len(nodes)]

keys = [f"object-{i}" for i in range(1000)]
before = [mod_placement(k, ["n1", "n2", "n3", "n4"])       for k in keys]
after  = [mod_placement(k, ["n1", "n2", "n3", "n4", "n5"]) for k in keys]

moved = sum(1 for b, a in zip(before, after) if b != a)
print(f"adding 1 node reshuffles {moved}/{len(keys)} keys ({moved/len(keys):.0%})")


adding 1 node reshuffles 790/1000 keys (79%)


### Better: consistent hashing with virtual nodes 💍

Imagine a ring of hash values. Each storage node hashes to several
points on the ring ("virtual nodes" or *vnodes*). A key hashes to
one point on the ring, and we walk clockwise until we hit a node.

Adding a node only takes over the slices closest to its vnodes — we
move about `1/N` of the keys. Vnodes smooth the distribution so no
one node ends up with a giant arc.


In [3]:
from bisect import bisect_right
import hashlib

class ConsistentHashRing:
    def __init__(self, vnodes_per_node: int = 100):
        self.vnodes_per_node = vnodes_per_node
        self.ring: list = []            # sorted list of (hash, node)

    @staticmethod
    def _h(s: str) -> int:
        return int(hashlib.md5(s.encode()).hexdigest(), 16)

    def add(self, node: str):
        for i in range(self.vnodes_per_node):
            self.ring.append((self._h(f"{node}#{i}"), node))
        self.ring.sort()

    def remove(self, node: str):
        self.ring = [(h, n) for h, n in self.ring if n != node]

    def node_for(self, key: str) -> str:
        if not self.ring:
            raise RuntimeError("empty ring")
        h = self._h(key)
        hashes = [x[0] for x in self.ring]
        idx = bisect_right(hashes, h) % len(self.ring)
        return self.ring[idx][1]


keys = [f"object-{i}" for i in range(1000)]
ring = ConsistentHashRing()
for n in ["n1", "n2", "n3", "n4"]:
    ring.add(n)

before = [ring.node_for(k) for k in keys]
ring.add("n5")
after  = [ring.node_for(k) for k in keys]

moved = sum(1 for b, a in zip(before, after) if b != a)
print(f"adding 1 node reshuffles {moved}/{len(keys)} keys ({moved/len(keys):.0%})")
print("~1/N of keys moved — not ~all of them.")


adding 1 node reshuffles 236/1000 keys (24%)
~1/N of keys moved — not ~all of them.


Real systems go further: they pick **R different nodes** from the
ring for each key (one per replica or shard) and make sure they're in
different failure domains. That's how erasure-coded shards get
spread across racks/zones without a central coordinator.


## 3️⃣ Erasure coding — intuition with XOR

Erasure coding = clever math that lets any `k` of `k+m` shards
rebuild the object. Real systems use **Reed–Solomon** over a finite
field, which is heavy math. But `k=2, m=1` has an intuitive form:
**parity = XOR of the data shards**. If one of the three is lost we
can recover it by XOR-ing the other two.

*(Why?* `a ⊕ b = p` ⇒ `a ⊕ p = b` ⇒ `b ⊕ p = a`. Try it!)


In [4]:
def xor(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))

data = b"hello world!!!!"
if len(data) % 2:
    data += b"\x00"
half = len(data) // 2
d1, d2 = data[:half], data[half:]
p = xor(d1, d2)

print("d1:", d1)
print("d2:", d2)
print("p :", p)

# Lose d2 -> rebuild from d1 XOR p
recovered_d2 = xor(d1, p)
print("recovered d2 == d2?", recovered_d2 == d2)
print("full:", d1 + recovered_d2)

# Lose d1 -> rebuild from d2 XOR p
recovered_d1 = xor(d2, p)
print("recovered d1 == d1?", recovered_d1 == d1)


d1: b'hello wo'
d2: b'rld!!!!\x00'
p : b'\x1a\t\x08MN\x01Vo'
recovered d2 == d2? True
full: b'hello world!!!!\x00'
recovered d1 == d1? True


To go beyond a single failure you need real Reed–Solomon (any `m`
failures tolerated), but the mental model is the same. In practice
you use a library — the cost per GB to encode/decode is tiny.


## 4️⃣ Lifecycle policies

Most objects are hot for a few days, then almost nobody touches them.
S3-style **storage classes** (Standard → Infrequent Access → Glacier)
trade latency for dollars. A **lifecycle policy** is a rule like:

> *Move to cold storage after 30 days; delete after 1 year.*

Here's the whole engine in ~15 lines.


In [5]:
from datetime import datetime, timedelta, timezone

class LifecyclePolicy:
    def __init__(self, to_cold_after_days: int, expire_after_days: int):
        self.to_cold = timedelta(days=to_cold_after_days)
        self.expire  = timedelta(days=expire_after_days)

    def action(self, created_at: datetime, now: datetime | None = None) -> str:
        now = now or datetime.now(timezone.utc)
        age = now - created_at
        if age >= self.expire:
            return "delete"
        if age >= self.to_cold:
            return "move_to_cold"
        return "keep_hot"

p = LifecyclePolicy(to_cold_after_days=30, expire_after_days=365)
now = datetime.now(timezone.utc)
for days in [1, 31, 200, 400]:
    created = now - timedelta(days=days)
    print(f"age {days:>3}d -> {p.action(created, now)}")


age   1d -> keep_hot
age  31d -> move_to_cold
age 200d -> move_to_cold
age 400d -> delete


A background worker (think cron + queue) iterates through the
metadata store, asks the policy what to do, and performs the action.
This is one of those places where **separating metadata from data**
pays off again — we don't need to scan petabytes to decide what to
move, we just walk a database table.


## 🧭 Closing thoughts

The simple theme across all three notebooks:

- **Small metadata plane, enormous data plane**, each with the tech
  that fits it.
- **Erasure coding** for cheap durability.
- **Multipart upload** for big objects; **presigned URLs** for safe
  sharing; **consistent hashing** for smooth growth; **lifecycle
  policies** for cost control.
- Every feature in a real S3 clone comes from a problem you can see
  break in ~50 lines of Python. Try modifying the code above — e.g.
  make the ring use only 1 vnode per node and watch the
  reshuffle-percentage go up.
